# Rules in the Tools: a Netomi Agent with Live Search on AlloyDB

## Section 1 - Backend Setup

In [ ]:
from first_available.db import terraform_env

terraform_env()
# tee, not a redirect: snap-installed Terraform can't write to a redirected file.
!terraform -chdir=terraform init -input=false -no-color 2>&1 | tee -a terraform/terraform.log \
  | grep --line-buffered -E "successfully initialized|^Error|│"
!terraform -chdir=terraform apply -auto-approve -input=false -no-color 2>&1 | tee -a terraform/terraform.log \
  | grep --line-buffered -E "Creation complete|Apply complete|^Error|│"
!terraform -chdir=terraform output -no-color

## Section 2 - Load the Catalog

In [ ]:
from first_available.db import get_connection, run_sql_file, query, print_table

conn = get_connection()
for name in ["schema.sql", "catalog.sql", "reset.sql", "agent_functions.sql"]:
    run_sql_file(conn, f"assets/sql/{name}")
print_table(query(conn, """SELECT name, city, service_zips, certified_brands, rating, profile
                         FROM technicians ORDER BY city, rating DESC"""),
            "Technicians in the catalog, by city and rating:")
print_table(query(conn, "SELECT part_number, name, brand, models, price, stock FROM parts LIMIT 6"),
            "The first 6 of the 36 parts in the catalog:")

## Section 3 - AlloyDB as a Search Engine

In [ ]:
from first_available.search import compare_search, technicians_for

for text in ["KS-PMP-410", "the rubber strip around my dishwasher door is torn", "KD-412 door seal"]:
    print_table(compare_search(conn, text), f'Parts search for "{text}": top 3 by keyword, vector and hybrid:')
for issue in ["water is sitting in the bottom of my dishwasher and it shows E24",
              "my oven takes forever to heat up and burns the bottom of cakes"]:
    print_table(technicians_for(conn, "80202", "Northfield", issue),
                f'Northfield technicians for 80202, ranked for the issue "{issue}":')

## Section 4 - The Agent's Leash: a Three-Tool MCP Server

In [ ]:
import os
from first_available.leash import leash_report
from first_available.mcp_client import start_server, start_tunnel, list_tools, call_tool, status_without_token
from first_available.search import technicians_table

run_sql_file(conn, "assets/sql/mcp_grants.sql")
print_table(leash_report(conn), "Leash report: what the MCP server's database user, mcp_search, can and can't do:")
start_server()
MCP_URL, TOKEN = start_tunnel(), os.environ["MCP_AUTH_TOKEN"]
print("MCP URL for Netomi:", MCP_URL)
print("Tools:", await list_tools(MCP_URL, TOKEN))
print_table(technicians_table(await call_tool(MCP_URL, TOKEN, "search_technicians",
    {"zip_code": "80202", "brand": "Northfield", "issue": "dishwasher shows E24"})),
    "search_technicians, called through ngrok with the bearer token (80202, Northfield, E24):")
print("Without the token:", status_without_token(MCP_URL))

## Section 5 - Configure the Netomi Agent

In [ ]:
import json

netomi_mcp = {"summit_home_services": {
    "transport": "streamable_http",
    "url": MCP_URL,
    "headers": {"Authorization": f"Bearer {TOKEN}", "User-Agent": "netomi"},
}}
print(json.dumps(netomi_mcp, indent=2))

## Section 6 - Talk to the Agent from Code

In [ ]:
import uuid
from first_available.netomi import get_token, send_message
from first_available.conversation import load_scenario, run_scenario, take_earliest_slot, bookings

token = get_token()
reply = send_message(token, f"demo-{uuid.uuid4()}", "hello")
print('The agent\'s reply to "hello" through the Netomi Chat API:\n')
print(f"Agent: {reply['text']}")

## Section 7 - Conversation 1: Diagnose and Book

In [ ]:
run_sql_file(conn, "assets/sql/reset.sql")
run_scenario(load_scenario("assets/scenarios/1-diagnose-and-book.yaml"))
print_table(bookings(conn), "Bookings in the database (expect one, for Jordan Smith, with the technician they chose):")

## Section 8 - Conversation 2: Slot Taken, Agent Recovers

In [ ]:
run_sql_file(conn, "assets/sql/reset.sql")
scenario = load_scenario("assets/scenarios/2-slot-taken.yaml")
run_scenario(scenario, after_agent_reply=take_earliest_slot(conn))
print_table(bookings(conn), "Bookings in the database (expect Another Customer in the first slot, Casey Rivera in the next):")

## Section 9 - Conversation 3: Find the Right Part

In [ ]:
run_sql_file(conn, "assets/sql/reset.sql")
transcript = run_scenario(load_scenario("assets/scenarios/3-find-the-part.yaml"))
agent_said = " ".join(text for speaker, text in transcript if speaker == "Agent")
print("The agent named the right part (KS-GSK-412):", "KS-GSK-412" in agent_said)

## Section 10 - Conversation 4: Staying Safe

In [ ]:
run_sql_file(conn, "assets/sql/reset.sql")
run_scenario(load_scenario("assets/scenarios/4-prompt-injection.yaml"))
print_table(query(conn, """SELECT (SELECT count(*) FROM bookings) AS bookings,
                       (SELECT count(*) FROM technicians) AS technicians,
                       (SELECT count(*) FROM parts) AS parts"""),
            "Row counts after the injection attempts (expect them unchanged: 0 bookings, 12 technicians, 36 parts):")

## Section 11 - Conversation 5: Escalation to a Human

In [ ]:
run_sql_file(conn, "assets/sql/reset.sql")
run_scenario(load_scenario("assets/scenarios/5-escalation.yaml"))
print_table(bookings(conn), "Bookings in the database (expect none: the agent escalates instead):")

## Section 12 - Cleanup

In [ ]:
from first_available.db import terraform_env

terraform_env()
!pkill -f "[f]irst_available.mcp_server" && echo "Stopped the MCP server." || echo "MCP server was not running."
!pkill -f "[n]grok http" && echo "Stopped ngrok." || echo "ngrok was not running."
!terraform -chdir=terraform destroy -auto-approve -input=false -no-color 2>&1 | tee -a terraform/terraform.log \
  | grep --line-buffered -E "Destruction complete|Destroy complete|^Error|│"
remaining = !terraform -chdir=terraform state list
print("Resources Terraform still manages (should be none):", *remaining, sep="\n")